In [ ]:
import pandas as pd
import numpy as np
import os
import scanpy as sc
import anndata as ad

In [ ]:
dataset = "Melanoma_Oct2522"

In [ ]:
dfs = []
path = f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/ManualLabels/{dataset}/quantification"
for df in os.listdir(path):
    if df.endswith(".csv"):
        image_name = df.split(".")[0]
        df = pd.read_csv(os.path.join(path, df))
        df['sample_id'] = image_name
        df = df.rename(columns={"CellID": "cell_id", "X_centroid": "x", "Y_centroid": "y"})
        dfs.append(df)
        df_concat = pd.concat(dfs, ignore_index=True)



In [ ]:
df_concat

In [ ]:
df_labels = pd.read_csv(f'/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/CellTuneDepot/ManualLabels/{dataset}/Tables/Labels.csv')
df_labels = df_labels.rename(columns={"cellID": "cell_id", "image": "sample_id", "class": "cell_type"})

In [ ]:
df_labels

In [ ]:
df_united = pd.merge(df_concat, df_labels, on=["cell_id", "sample_id"], how="inner")

In [ ]:
df_united

In [ ]:
df_united.columns

In [ ]:
cols = df_united.columns.tolist()
cols.remove('cell_id')
chga_idx = cols.index('Orientation')
cols.insert(chga_idx + 1, 'cell_id')
df_united = df_united[cols]


In [ ]:
df_united['cell_type'].value_counts()

In [ ]:
X_columns = df_united.columns[:df_united.columns.get_loc('x')]
obs_columns = df_united.columns[df_united.columns.get_loc('x'):]
adata = ad.AnnData(
    X=df_united[X_columns],
    obs=df_united[obs_columns],
    var=pd.DataFrame(index=X_columns)
)

In [ ]:
adata

In [ ]:
sc.pl.matrixplot(adata, var_names=adata.var_names, groupby='cell_type', cmap='coolwarm', use_raw=False, standard_scale='var')

In [ ]:
df_united['cell_type'] = df_united['cell_type'].replace({"TumorEpithelial": "Cancer", "CD4T": "CD4+_T_cell", "CD8T": "CD8+_T_cell", "Bcell": "B_cell", "Plasma": "Plasma_cell", "Unidentified": "undefined", 
                                           "ImmuneOther": "undefined", "Mast": "Mast_cell", "CD3T": "T_cell", "DC": "Dendritic_cell", "Neuron": "Neuronal", "SMV": "vSMC",
                                           "Tumor": "Cancer","Endocrine": "Endocrine_cell", "Goblet": "Goblet_cell", "Paneth": "Paneth_cell", "ActivatedFibro": "Fibroblast", 'Garbage': "undefined",
                                           "MyoFibro": "Myofibroblasts", 'NK': "NK_cell", "NKT": "NK_T_cell", "Tcell": "T_cell", "Tumor_BCAT": "Cancer", 'HLM': "Macrophage"})

In [ ]:
df_united.to_csv(os.path.join(path, f"{dataset}_quantification.csv"), index=False)

In [ ]:
dataset = 'Breast_Jun0524'

In [ ]:
path = f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/{dataset}/quantification/processed/{dataset}_quantification.csv"

In [ ]:
df = pd.read_csv(path)

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df['cell_type'].nunique()

In [ ]:
df['cell_type'].value_counts()

In [ ]:
df['sample_id'].unique()

In [ ]:
df['cell_type'] = df['cell_type'].replace({'vSMC': 'VSMC', 'Stroma': 'Stromal'})
df['level_2_cell_type'] = df['cell_type']
df['level_2_cell_type'] = df['level_2_cell_type'].replace({'CD4+_T_cell':'Lymphoid_immune', 'CD8+_T_cell': 'Lymphoid_immune','B_cell': 'Lymphoid_immune', 'Treg': 'Lymphoid_immune', 
                                                           'NK_T_cell': 'Lymphoid_immune', 'NK_cell': 'Lymphoid_immune', 'T_cell': 'Lymphoid_immune', 'Myeloid': 'Myeloid_immune', 'Macrophage': 'Myeloid_immune',
                                                           'Neutrophil': 'Myeloid_immune', 'Endothelial': 'Vascular', 'VSMC': 'Vascular', 'Myofibroblasts': 'Fibroblast', 'T_cell': 'Lymphoid_immune', 'Plasma_cell': 'Lymphoid_immune',
                                                           'Goblet_cell': 'Epithelial', 'Paneth_cell': 'Epithelial', 'Endocrine_cell': 'Epithelial', 'Mast_cell': 'Myeloid_immune', 'Dendritic_cell': 'Myeloid_immune',
                                                           'Lymphatic': 'Vascular', 'Monocyte': 'Myeloid_immune', 'APC': 'Myeloid_immune'})

In [ ]:
df['level_2_cell_type'].value_counts()

In [ ]:
df['level_1_cell_type'] = df['level_2_cell_type']
df['level_1_cell_type'] = df['level_1_cell_type'].replace({'Lymphoid_immune': 'Immune', 'Myeloid_immune': 'Immune', 'Vascular': 'Stromal', 'Fibroblast': 'Stromal'})

In [ ]:
df['level_1_cell_type'].value_counts()

In [ ]:
df.to_csv(os.path.join(path), index=False)